# Exploratory Data Analysis of the Vitamin D Transcriptomic Subset

This notebook presents the exploratory data analysis (EDA) of a curated subset of the LINCS L1000 dataset, 
focusing on transcriptional responses to Vitamin D and its analogs in human cell lines. 

The subset was generated from the LINCS2020 release and restricted to:
- **Compounds**: Vitamin D and related analogs (e.g., calcitriol, calcipotriol, paricalcitol, maxacalcitol, ercalcitriol, tacalcitol, seocalcitol).
- **Cell lines**: Five representative human lines (PC3, MCF7, A549, U2OS, HA1E).
- **Perturbation time**: 24 hours.

The aim of this analysis is to:
1. Characterize the distribution of signatures across compounds and cell lines.  
2. Assess the overall structure of the expression matrix.  
3. Explore transcriptomic similarities via dimensionality reduction and clustering.  
4. Evaluate signature quality using available metrics.  

These steps provide the foundation for downstream modeling and biological interpretation.

## Data Loading and Initial Setup

We start by loading the exported subset of the LINCS L1000 dataset, focusing on Vitamin D and its analogs.  
This includes the expression matrix (genes × signatures) and metadata files for signatures, compounds, and cell lines.  
All files are stored in CSV format, curated from the original LINCS2020 release.

In [ ]:
# Import core libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load subset metadata and expression files
cell_meta = pd.read_csv("../data/exports/subset_cell_lines_meta.csv")
cmp_meta = pd.read_csv("../data/exports/subset_compounds_meta.csv")
gene_meta = pd.read_csv("../data/exports/subset_genes_meta.csv")
sig_meta = pd.read_csv("../data/exports/subset_signatures_meta.csv")
exp_df = pd.read_csv("../data/exports/subset_expression_wide_gene_id.csv", index_col=0)

# Quick checks on dimensions
print("Cell lines:", cell_meta.shape)
print("Compounds:", cmp_meta.shape)
print("Genes:", gene_meta.shape)
print("Signatures:", sig_meta.shape)
print("Expression matrix:", exp_df.shape)

# Preview first rows
display(cell_meta.head())
display(cmp_meta.head())
display(sig_meta.head())
display(exp_df.iloc[:5, :5])


#### Conclusion
The dataset comprises **422 transcriptional signatures**, derived from **12 Vitamin D–related compounds** tested across **5 distinct human cell lines**, with expression values for **12,328 genes**.  
This balanced yet limited subset provides a manageable foundation for subsequent exploratory and comparative analyses.

## Signature Distribution by Compound and Cell Line — setup

We quantify how many signatures are available per compound and per cell line to detect potential imbalance that could bias downstream analyses.

In [ ]:
# Build tidy summary tables for signatures per compound and per cell line

# Map pert_id -> compound name
pert_lookup = cmp_meta[['pert_id', 'cmap_name']].drop_duplicates()

# --- Counts by compound (pert_id) ---
sig_by_pert = (
    sig_meta
    .groupby('pert_id', as_index=False)
    .agg(n_signatures=('sig_id', 'count'))
    .merge(pert_lookup, on='pert_id', how='left')
    .assign(cmap_name=lambda d: d['cmap_name'].fillna(d['pert_id']))
    .sort_values('n_signatures', ascending=False)
    .reset_index(drop=True)
)

# Count signatures grouped by compound name (cmap_name)
sig_by_cmap = (
    sig_meta
    .merge(cmp_meta[['pert_id', 'cmap_name']], on='pert_id', how='left')
    .groupby('cmap_name', as_index=False)
    .agg(n_signatures=('sig_id', 'count'))
    .sort_values('n_signatures', ascending=False)
    .reset_index(drop=True)
)

display(sig_by_cmap)


# --- Counts by cell line ---
sig_by_cell = (
    sig_meta
    .groupby('cell_id', as_index=False)
    .agg(n_signatures=('sig_id', 'count'))
    .sort_values('n_signatures', ascending=False)
    .reset_index(drop=True)
)

# Display summary tables
display(sig_by_pert)
display(sig_by_cell)


In [ ]:
# Barplot: signatures per compound
plt.figure(figsize=(10, 5))
sns.barplot(
    data=sig_by_pert,
    x='n_signatures',
    y='cmap_name',
    order=sig_by_pert.sort_values('n_signatures', ascending=False)['cmap_name'],
    errorbar=None,
    color="steelblue"    
)
plt.xlabel('Number of signatures')
plt.ylabel('Compound (cmap_name)')
plt.title('Signatures per compound')
plt.tight_layout()
plt.show()

# Barplot: signatures per cell line
plt.figure(figsize=(7, 4))
sns.barplot(
    data=sig_by_cell,
    x='cell_id',
    y='n_signatures',
    order=sig_by_cell.sort_values('n_signatures', ascending=False)['cell_id'],
    errorbar=None,
    color="steelblue"    
)
plt.xlabel('Cell line')
plt.ylabel('Number of signatures')
plt.title('Signatures per cell line')
plt.tight_layout()
plt.show()
